# Customer Churn Prediction for a Subscription Business

This notebook explores customer churn, compares classification models, and connects predictions to retention strategy.

**Business question:** Which customers are most likely to cancel their subscription, and what factors are associated with higher churn risk?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score,
    f1_score, accuracy_score, ConfusionMatrixDisplay,
    RocCurveDisplay
)

df = pd.read_csv("../data/subscription_churn.csv")
df.head()


## 1. Data Quality and Churn Rate


In [ ]:
print(df.shape)
print(df.isna().sum())
print(df["churn"].value_counts(normalize=True).rename("share"))


## 2. Exploratory Analysis


In [ ]:
contract_churn = (
    df.groupby("contract_type")["churn"]
      .mean()
      .sort_values(ascending=False)
)

contract_churn.plot(kind="bar", title="Churn Rate by Contract Type")
plt.ylabel("Churn Rate")
plt.tight_layout()
plt.show()


In [ ]:
satisfaction_churn = (
    df.groupby("satisfaction_score")["churn"]
      .mean()
)

satisfaction_churn.plot(kind="bar", title="Churn Rate by Satisfaction Score")
plt.ylabel("Churn Rate")
plt.tight_layout()
plt.show()


In [ ]:
autopay_churn = df.groupby("autopay")["churn"].mean()
autopay_churn.plot(kind="bar", title="Churn Rate by Autopay Status")
plt.ylabel("Churn Rate")
plt.tight_layout()
plt.show()


## 3. Prepare Features


In [ ]:
X = df.drop(columns=["customer_id", "churn"])
y = df["churn"]

categorical = [
    "contract_type", "payment_method", "autopay",
    "discount_active", "region"
]

numeric = [
    "tenure_months", "monthly_charge", "support_tickets_90d",
    "weekly_usage_hours", "num_products", "late_payments_12m",
    "satisfaction_score"
]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric),
    ("cat", categorical_pipe, categorical)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=42
)


## 4. Compare Models


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1500,
        class_weight="balanced"
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=5,
        random_state=42,
        class_weight="balanced"
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    )
}

results = []
trained_models = {}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)

    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, proba)
    })

    trained_models[name] = pipe

results_df = pd.DataFrame(results).sort_values(
    "ROC-AUC",
    ascending=False
)

results_df


## 5. Evaluate the Best Model


In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_name]

print("Best model:", best_name)

ConfusionMatrixDisplay.from_estimator(
    best_model,
    X_test,
    y_test
)
plt.title(f"Confusion Matrix - {best_name}")
plt.show()

RocCurveDisplay.from_estimator(
    best_model,
    X_test,
    y_test
)
plt.title(f"ROC Curve - {best_name}")
plt.show()


## 6. Create Customer Risk Segments


In [ ]:
scored = X_test.copy()
scored["actual_churn"] = y_test.values
scored["churn_probability"] = best_model.predict_proba(X_test)[:, 1]

scored["risk_segment"] = pd.cut(
    scored["churn_probability"],
    bins=[-0.01, 0.35, 0.65, 1.0],
    labels=["Low", "Medium", "High"]
)

scored[
    [
        "contract_type",
        "tenure_months",
        "support_tickets_90d",
        "satisfaction_score",
        "churn_probability",
        "risk_segment"
    ]
].sort_values(
    "churn_probability",
    ascending=False
).head(15)


## 7. Business Interpretation

The model is most useful when its output is translated into action.

A retention team could use the churn probability to:

- prioritize high-risk customers for outreach;
- review recurring support issues;
- test targeted contract or discount offers;
- encourage autopay or annual contracts where appropriate;
- monitor customers with declining satisfaction or usage.

A real production system would also require fairness, privacy, model monitoring, and controlled experimentation before retention interventions are automated.
